In [1]:
from newsapi import NewsApiClient
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install setuptools
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews
import json


In [2]:
# tech stocks, pharma stocks, oil stocks, tobacco stocks
tickers = ["AAPL", "MSFT", "PG", "XOM", "BTI"]

# News Data Download
- newsapi has a limit of 2024-11-01 onwards
- pygoognews https://github.com/kotartemiy/pygooglenews
- Some other packages to try out: https://www.newscatcherapi.com/blog/python-web-scraping-libraries-to-mine-news-data

In [3]:
# Init
newsapi = NewsApiClient(api_key='bcd7f19d8f6744d4b6c3bec00f32e1c6')

# /v2/top-headlines
top_headlines = newsapi.get_top_headlines(q='apple',
                                          sources='bbc-news,the-verge',
                                          language='en')

# /v2/everything
all_articles = newsapi.get_everything(q='apple',
                                      sources='bbc-news,the-verge',
                                      domains='bbc.co.uk,techcrunch.com',
                                      from_param='2024-11-02',
                                      to='2024-12-01',
                                      language='en',
                                      sort_by='relevancy',
                                      page=2)

# /v2/top-headlines/sources
sources = newsapi.get_sources()

In [4]:
gn = GoogleNews(lang = 'en')

news_dfs = []
for ticker in tickers:
    top = gn.search(ticker)
    entries = top["entries"]
    df_temp = clean_goog_news(entries)
    news_dfs += [df_temp.copy()]


(99, 3)
(99, 3)
(99, 3)
(100, 3)
(97, 3)


In [5]:
news_dfs[0].sample(10)

,date,title,source
43,"Wed, 27 Nov 2024 18:38:00 GMT","Noteworthy Wednesday Option Activity: AAPL, UL...",https://www.nasdaq.com
40,"Sun, 24 Nov 2024 13:22:44 GMT",Is Apple Inc. (AAPL) Still a Key Fixture in Wa...,https://finance.yahoo.com
83,"Thu, 28 Nov 2024 09:05:00 GMT","Planned Solutions Inc. Purchases 3,472 Shares ...",https://www.marketbeat.com
21,"Thu, 07 Nov 2024 08:00:00 GMT",Jim Cramer Admits Apple (AAPL) is a ‘Changing ...,https://finance.yahoo.com
34,"Thu, 28 Nov 2024 04:44:00 GMT",Where are the Opportunities in (AAPL) - Stock ...,https://news.stocktradersdaily.com
2,"Fri, 29 Nov 2024 00:02:04 GMT",Apple Inc (AAPL) DCF Valuation: Is The Stock U...,https://acquirersmultiple.com
30,"Sun, 01 Dec 2024 08:11:45 GMT",Apple Inc. (NASDAQ:AAPL) Stake Trimmed by Vale...,https://www.marketbeat.com
79,"Fri, 29 Nov 2024 08:27:32 GMT","Worth Asset Management LLC Buys 4,609 Shares o...",https://www.marketbeat.com
78,"Thu, 28 Nov 2024 17:02:11 GMT",Apple (NASDAQ:AAPL) Shares Down 0.1% - What's ...,https://www.marketbeat.com
16,"Mon, 25 Nov 2024 13:29:09 GMT",Is Apple Inc. (AAPL) the Best Stock to Invest ...,https://finance.yahoo.com


# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [6]:
stocks_dfs = []

for ticker in tickers:
    stock = yf.download(ticker, start='2023-01-01', end='2024-12-01')
    stock_cleaned = clean_scraped_data(stock)
    stocks_dfs += [stock_cleaned.copy()]

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [7]:
ticker

'BTI'

In [8]:
stocks_dfs[0]

Price,date,Adj Close,Close,High,Low,Open,Volume
0,2023-01-03,123.768456,125.070000,130.899994,124.169998,130.279999,112117500
1,2023-01-04,125.045044,126.360001,128.660004,125.080002,126.889999,89113600
2,2023-01-05,123.718987,125.019997,127.769997,124.760002,127.129997,80962700
3,2023-01-06,128.271103,129.619995,130.289993,124.889999,126.010002,87754700
4,2023-01-09,128.795593,130.149994,133.410004,129.889999,130.470001,70790800
...,...,...,...,...,...,...,...
476,2024-11-22,229.869995,229.869995,230.720001,228.059998,228.059998,38168300
477,2024-11-25,232.869995,232.869995,233.250000,229.740005,231.460007,90152800
478,2024-11-26,235.059998,235.059998,235.570007,233.330002,233.330002,45986200
479,2024-11-27,234.929993,234.929993,235.690002,233.809998,234.470001,33498400


# Save Data

In [9]:
with pd.ExcelWriter('Data/news_data.xlsx') as writer:
    for i, news_df in enumerate(news_dfs):
        news_df.to_excel(writer, sheet_name=tickers[i], index=False)

# Close the ExcelWriter object
writer.save()

c:\Users\Ethelda\anaconda3\lib\site-packages\xlsxwriter\workbook.py:336: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")


In [10]:
with pd.ExcelWriter('Data/stocks_data.xlsx') as writer:
    for i, stock_df in enumerate(stocks_dfs):
        stock_df.to_excel(writer, sheet_name=tickers[i], index=False)

# Close the ExcelWriter object
writer.save()

c:\Users\Ethelda\anaconda3\lib\site-packages\xlsxwriter\workbook.py:336: UserWarning: Calling close() on already closed file.
  warn("Calling close() on already closed file.")
